# CS 5588 Challenge 1 — AI-Guided Job Search Application

**Data Science Capstone • Fall 2026**

This notebook is a teaching starter for comparing:

1. **Human design** — define the problem, data, baseline, and tests manually.
2. **AI-guided design** — use an agentic AI tool to propose and improve code.
3. **Human–AI co-design** — inspect, constrain, verify, and refine the AI output.
4. **Feedback & refinement** — use observed errors and user feedback to improve the system.

### Toolchain demonstrated
- **Google Colab / Jupyter Notebook** — executable notebook workflow
- **GitHub** — version control, issues, branches, commits, pull requests
- **Hugging Face** — semantic embeddings and an optional `smolagents` example
- **Google Antigravity** — agentic coding prompt/activity (plan → edit → execute → verify)

> **Class rule:** AI may generate code, but your team owns every line it submits. Run it, test it, explain it, and commit it.

## 0. Learning goals

By the end of this notebook, you should be able to:

- represent a job seeker and job postings as structured data;
- build a transparent baseline job matcher;
- search, filter, score, and rank jobs;
- explain why a job matched and identify gaps;
- use Hugging Face embeddings for semantic matching;
- understand a simple agentic **plan → tool → observe → revise → verify** loop;
- use an AI coding agent such as Antigravity to improve a bounded part of the notebook;
- track manual vs. AI-guided development with GitHub and evaluation evidence.

We use **synthetic job postings** in the starter notebook so the class can focus on reproducibility and system design.

In [ ]:
# 1. Environment check: works in Jupyter and Google Colab

import sys, platform, os, json, re, math, time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Running in Google Colab:", IN_COLAB)

if IN_COLAB:
    print("Tip: Runtime > Change runtime type if you want GPU support for larger models.")
else:
    print("Tip: This notebook also runs locally in Jupyter.")

## 2. Optional package setup

The manual baseline uses common Python/data-science libraries.

The Hugging Face semantic section uses:

- `sentence-transformers`
- `transformers`
- optionally `smolagents`

In **Google Colab**, uncomment the installation line below.

In [ ]:
# Uncomment in a fresh Colab runtime:
# %pip -q install pandas scikit-learn sentence-transformers transformers huggingface_hub smolagents

import pandas as pd
import numpy as np

pd.set_option("display.max_colwidth", 120)

# Part A — Human Design First

## 3. Define the user profile

Start with a transparent, explicit representation of the user's qualifications and preferences.

For this exercise, the profile contains:

- skills;
- years of experience;
- interests;
- preferred locations;
- remote preference;
- target job titles.

In [ ]:
user_profile = {
    "name": "Sample Student",
    "skills": [
        "python", "sql", "pandas", "scikit-learn",
        "data visualization", "machine learning", "git"
    ],
    "years_experience": 2,
    "interests": ["data science", "healthcare", "ai"],
    "preferred_locations": ["Kansas City, MO", "Remote"],
    "remote_ok": True,
    "target_titles": ["Data Scientist", "Data Analyst", "ML Engineer"]
}

user_profile

## 4. Create a small job-posting dataset

For the kickoff, use a controlled mini-dataset rather than scraping arbitrary sites.

Each posting contains title/company, skills, description, location, experience, and work arrangement.

Later, your team can replace this with an instructor-approved dataset or API.

In [ ]:
jobs = [
    {
        "job_id": "J001",
        "title": "Junior Data Scientist",
        "company": "HealthAI Labs",
        "location": "Kansas City, MO",
        "work_mode": "Hybrid",
        "min_experience": 1,
        "required_skills": ["python", "sql", "pandas", "machine learning"],
        "preferred_skills": ["healthcare", "scikit-learn", "git"],
        "description": "Build predictive models and analytics pipelines for healthcare data using Python, SQL, pandas, and machine learning."
    },
    {
        "job_id": "J002",
        "title": "Data Analyst",
        "company": "Metro Analytics",
        "location": "Kansas City, MO",
        "work_mode": "On-site",
        "min_experience": 1,
        "required_skills": ["sql", "excel", "data visualization"],
        "preferred_skills": ["python", "tableau"],
        "description": "Analyze business data, create SQL reports, dashboards, and visualizations, and communicate findings to stakeholders."
    },
    {
        "job_id": "J003",
        "title": "Machine Learning Engineer",
        "company": "VectorWorks AI",
        "location": "Remote",
        "work_mode": "Remote",
        "min_experience": 3,
        "required_skills": ["python", "pytorch", "docker", "machine learning"],
        "preferred_skills": ["aws", "mlops", "git"],
        "description": "Develop production machine learning systems, model training pipelines, APIs, containers, and cloud deployments."
    },
    {
        "job_id": "J004",
        "title": "Research Data Scientist",
        "company": "BioDiscovery Institute",
        "location": "St. Louis, MO",
        "work_mode": "Hybrid",
        "min_experience": 2,
        "required_skills": ["python", "statistics", "machine learning"],
        "preferred_skills": ["healthcare", "nlp", "pandas"],
        "description": "Apply statistical learning and AI methods to biomedical research datasets and collaborate with domain scientists."
    },
    {
        "job_id": "J005",
        "title": "Business Intelligence Analyst",
        "company": "Civic Data Group",
        "location": "Remote",
        "work_mode": "Remote",
        "min_experience": 2,
        "required_skills": ["sql", "power bi", "data visualization"],
        "preferred_skills": ["python", "data modeling"],
        "description": "Build dashboards, semantic models, SQL transformations, and KPI reporting for distributed business teams."
    },
    {
        "job_id": "J006",
        "title": "AI Software Intern",
        "company": "Agentic Systems",
        "location": "Remote",
        "work_mode": "Remote",
        "min_experience": 0,
        "required_skills": ["python", "git"],
        "preferred_skills": ["llm", "hugging face", "agents"],
        "description": "Prototype AI applications using Python, open-source models, GitHub workflows, and agentic development tools."
    },
    {
        "job_id": "J007",
        "title": "Senior Data Scientist",
        "company": "FinModel",
        "location": "Chicago, IL",
        "work_mode": "Hybrid",
        "min_experience": 5,
        "required_skills": ["python", "sql", "statistics", "machine learning"],
        "preferred_skills": ["spark", "aws", "leadership"],
        "description": "Lead modeling projects, mentor data scientists, and design scalable machine learning solutions for financial products."
    },
    {
        "job_id": "J008",
        "title": "Healthcare Data Analyst",
        "company": "CareMetrics",
        "location": "Remote",
        "work_mode": "Remote",
        "min_experience": 1,
        "required_skills": ["sql", "python", "data visualization"],
        "preferred_skills": ["healthcare", "pandas"],
        "description": "Analyze healthcare quality data with SQL and Python, prepare visual reports, and support clinical analytics teams."
    },
    {
        "job_id": "J009",
        "title": "Data Engineering Associate",
        "company": "CloudPipe",
        "location": "Kansas City, MO",
        "work_mode": "Hybrid",
        "min_experience": 2,
        "required_skills": ["python", "sql", "etl"],
        "preferred_skills": ["spark", "aws", "git"],
        "description": "Create ETL workflows, data quality checks, SQL transformations, and cloud-oriented data pipelines."
    },
    {
        "job_id": "J010",
        "title": "NLP Research Assistant",
        "company": "University AI Lab",
        "location": "Kansas City, MO",
        "work_mode": "On-site",
        "min_experience": 0,
        "required_skills": ["python", "machine learning"],
        "preferred_skills": ["nlp", "hugging face", "research"],
        "description": "Support experiments in natural language processing, machine learning, Hugging Face models, and research evaluation."
    }
]

jobs_df = pd.DataFrame(jobs)
jobs_df[["job_id", "title", "company", "location", "work_mode", "min_experience"]]

## 5. Basic job search: filter before ranking

A job-search application should distinguish:

1. **hard constraints** — conditions the user may not accept;
2. **soft preferences** — factors that affect rank but do not necessarily remove a job.

In [ ]:
def search_jobs(df, title_keyword=None, location=None, remote_only=False):
    result = df.copy()

    if title_keyword:
        result = result[result["title"].str.contains(title_keyword, case=False, na=False)]

    if location:
        result = result[result["location"].str.contains(location, case=False, na=False)]

    if remote_only:
        result = result[result["work_mode"].str.lower() == "remote"]

    return result.reset_index(drop=True)

print("Example 1: jobs containing 'Data'")
display(search_jobs(jobs_df, title_keyword="Data")[["job_id", "title", "company", "location"]])

print("\nExample 2: remote jobs")
display(search_jobs(jobs_df, remote_only=True)[["job_id", "title", "company", "location"]])

## 6. Transparent baseline matcher

We start with an explainable weighted score:

\[
\text{score} =
0.45(\text{skills}) +
0.25(\text{experience}) +
0.15(\text{interests}) +
0.15(\text{constraints})
\]

This is intentionally simple. The purpose is to make assumptions visible before asking AI to improve them.

In [ ]:
def normalize_token(x):
    return re.sub(r"\s+", " ", str(x).strip().lower())

def skill_match_score(user_skills, required_skills, preferred_skills=None):
    user = {normalize_token(s) for s in user_skills}
    required = {normalize_token(s) for s in required_skills}
    preferred = {normalize_token(s) for s in (preferred_skills or [])}

    required_score = len(user & required) / max(len(required), 1)
    preferred_score = len(user & preferred) / max(len(preferred), 1) if preferred else 0.0

    score = 0.8 * required_score + 0.2 * preferred_score
    return min(max(score, 0.0), 1.0)

def experience_match_score(user_years, min_years):
    if min_years <= 0:
        return 1.0
    return min(max(user_years / min_years, 0.0), 1.0)

def interest_match_score(interests, job):
    text = " ".join([
        job["title"], job["company"], job["description"],
        " ".join(job["required_skills"]),
        " ".join(job["preferred_skills"])
    ]).lower()

    if not interests:
        return 0.0

    hits = sum(normalize_token(i) in text for i in interests)
    score = hits / len(interests)
    return min(max(score, 0.0), 1.0)

def constraint_match_score(profile, job):
    location_match = job["location"] in profile["preferred_locations"]
    remote_match = profile["remote_ok"] and job["work_mode"].lower() == "remote"

    if location_match or remote_match:
        return 1.0

    return 0.25

def baseline_match(profile, job):
    skill = skill_match_score(
        profile["skills"],
        job["required_skills"],
        job["preferred_skills"]
    )
    exp = experience_match_score(profile["years_experience"], job["min_experience"])
    interest = interest_match_score(profile["interests"], job)
    constraints = constraint_match_score(profile, job)

    total = (
        0.45 * skill +
        0.25 * exp +
        0.15 * interest +
        0.15 * constraints
    )

    clamped = min(max(total, 0.0), 1.0)

    return {
        "score": round(clamped, 4),
        "skill_score": round(skill, 4),
        "experience_score": round(exp, 4),
        "interest_score": round(interest, 4),
        "constraint_score": round(constraints, 4)
    }


In [ ]:
baseline_rows = []

for job in jobs:
    result = baseline_match(user_profile, job)
    baseline_rows.append({
        "job_id": job["job_id"],
        "title": job["title"],
        "company": job["company"],
        **result
    })

baseline_ranked = (
    pd.DataFrame(baseline_rows)
    .sort_values("score", ascending=False)
    .reset_index(drop=True)
)

baseline_ranked

## 7. Explain a recommendation

A useful job matcher should show evidence, not only a number:

- matched required skills;
- missing required skills;
- matched preferred skills;
- experience comparison;
- location/work-mode evidence;
- final score.

In [ ]:
def explain_match(profile, job):
    user = {normalize_token(s) for s in profile["skills"]}
    required = {normalize_token(s) for s in job["required_skills"]}
    preferred = {normalize_token(s) for s in job["preferred_skills"]}

    scores = baseline_match(profile, job)

    matched_req = sorted(user & required)
    missing_req = sorted(required - user)
    matched_pref = sorted(user & preferred)

    score = scores["score"]
    if score >= 0.75:
        match_tier = "Strong Fit"
    elif score >= 0.50:
        match_tier = "Moderate Fit"
    else:
        match_tier = "Gap Warning"

    gap_summary = []
    if missing_req:
        gap_summary.append(f"Missing required skills: {', '.join(missing_req)}")
    if profile["years_experience"] < job["min_experience"]:
        gap = job["min_experience"] - profile["years_experience"]
        gap_summary.append(f"Experience gap of {gap} year(s)")

    return {
        "job": f'{job["title"]} @ {job["company"]}',
        "final_score": scores["score"],
        "match_tier": match_tier,
        "sub_scores": {
            "skill": scores["skill_score"],
            "experience": scores["experience_score"],
            "interest": scores["interest_score"],
            "constraints": scores["constraint_score"]
        },
        "matched_required": matched_req,
        "missing_required": missing_req,
        "matched_preferred": matched_pref,
        "experience": f'{profile["years_experience"]} years vs. {job["min_experience"]} required',
        "location_evidence": f'{job["location"]} / {job["work_mode"]}',
        "gap_summary": gap_summary if gap_summary else ["No critical gaps identified"]
    }

top_job_id = baseline_ranked.iloc[0]["job_id"]
top_job = next(j for j in jobs if j["job_id"] == top_job_id)

explain_match(user_profile, top_job)


## 8. Visible tests

A coding agent should not merely claim success. It should run tests and show outputs.

In [ ]:
def run_baseline_tests():
    tests = []

    s1 = skill_match_score(["python", "sql"], ["python", "sql"], [])
    tests.append(("perfect required-skill overlap", abs(s1 - 0.8) < 1e-9))

    s2 = skill_match_score(["python"], ["python", "sql"], [])
    tests.append(("missing one required skill lowers score", s2 < s1))

    e1 = experience_match_score(1, 4)
    e2 = experience_match_score(4, 4)
    tests.append(("experience threshold matters", e1 < e2))

    in_range = baseline_ranked["score"].between(0, 1).all()
    tests.append(("all final scores are in [0,1]", bool(in_range)))

    exp_res = explain_match(user_profile, top_job)
    req_keys = {"job", "final_score", "match_tier", "sub_scores", "matched_required", "missing_required", "matched_preferred", "experience", "location_evidence", "gap_summary"}
    tests.append(("explain_match returns full evidence schema", req_keys.issubset(exp_res.keys())))

    e_cap = experience_match_score(10, 3)
    tests.append(("experience score capped at 1.0 when candidate exceeds min_years", e_cap == 1.0))

    i_zero = interest_match_score([], top_job)
    tests.append(("empty interest list yields score 0.0 safely", i_zero == 0.0))

    j_test = jobs[0]
    p_full = {"skills": j_test["required_skills"], "years_experience": 5, "interests": [], "preferred_locations": [j_test["location"]], "remote_ok": True}
    p_part = {"skills": j_test["required_skills"][:-1], "years_experience": 5, "interests": [], "preferred_locations": [j_test["location"]], "remote_ok": True}
    tests.append(("missing required skill strictly lowers overall match score", baseline_match(p_part, j_test)["score"] < baseline_match(p_full, j_test)["score"]))

    return pd.DataFrame(tests, columns=["test", "passed"])

test_results = run_baseline_tests()
display(test_results)

assert test_results["passed"].all(), "At least one baseline test failed."
print("All baseline tests passed.")


# Part B — A Stronger Manual Baseline

## 9. TF-IDF semantic text matching

Exact skill overlap is transparent but brittle. A second baseline compares the profile text with each job using TF-IDF + cosine similarity.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

profile_text = " ".join(
    user_profile["skills"] +
    user_profile["interests"] +
    user_profile["target_titles"]
)

job_texts = [
    " ".join([
        j["title"],
        j["description"],
        " ".join(j["required_skills"]),
        " ".join(j["preferred_skills"])
    ])
    for j in jobs
]

vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
X = vectorizer.fit_transform([profile_text] + job_texts)

tfidf_scores = cosine_similarity(X[0:1], X[1:]).flatten()

tfidf_ranked = jobs_df[["job_id", "title", "company"]].copy()
tfidf_ranked["tfidf_similarity"] = np.round(tfidf_scores, 4)
tfidf_ranked = tfidf_ranked.sort_values("tfidf_similarity", ascending=False).reset_index(drop=True)

tfidf_ranked

# Part C — Hugging Face Example

## 10. Semantic matching with a Hugging Face model

Example model:

`sentence-transformers/all-MiniLM-L6-v2`

This section is optional if the package/model is unavailable.

> Semantic similarity should augment—not silently replace—transparent evidence such as explicit required skills and user constraints.

In [ ]:
hf_available = False

try:
    from sentence_transformers import SentenceTransformer
    hf_available = True
    print("sentence-transformers is available.")
except Exception as e:
    print("Hugging Face semantic section skipped.")
    print("Install with: %pip -q install sentence-transformers")
    print("Reason:", type(e).__name__)

In [ ]:
if hf_available:
    try:
        model_name = "sentence-transformers/all-MiniLM-L6-v2"
        model = SentenceTransformer(model_name)

        profile_embedding = model.encode([profile_text], normalize_embeddings=True)
        job_embeddings = model.encode(job_texts, normalize_embeddings=True)

        hf_scores = (profile_embedding @ job_embeddings.T).flatten()

        hf_ranked = jobs_df[["job_id", "title", "company"]].copy()
        hf_ranked["hf_semantic_similarity"] = np.round(hf_scores, 4)
        hf_ranked = hf_ranked.sort_values("hf_semantic_similarity", ascending=False).reset_index(drop=True)

        display(hf_ranked)
    except Exception as e:
        hf_available = False
        print("Could not load/run the Hugging Face embedding model.")
        print("This can happen if the model is not cached and the runtime has no internet access.")
        print("Reason:", type(e).__name__, str(e)[:180])
else:
    print("No model loaded; continue to the next section.")

## 11. Hybrid score: transparent baseline + semantic similarity

Teaching example:

\[
\text{hybrid} = 0.70(\text{baseline}) + 0.30(\text{semantic})
\]

These are teaching weights, not learned truth. Teams should justify or tune any weights they use.

In [ ]:
if hf_available:
    baseline_map = dict(zip(baseline_ranked["job_id"], baseline_ranked["score"]))
    semantic_map = dict(zip(hf_ranked["job_id"], hf_ranked["hf_semantic_similarity"]))

    hybrid = jobs_df[["job_id", "title", "company"]].copy()
    hybrid["baseline_score"] = hybrid["job_id"].map(baseline_map)
    hybrid["semantic_score"] = hybrid["job_id"].map(semantic_map)
    hybrid["hybrid_score"] = (
        0.70 * hybrid["baseline_score"] +
        0.30 * hybrid["semantic_score"]
    ).round(4)

    hybrid = hybrid.sort_values("hybrid_score", ascending=False).reset_index(drop=True)
    display(hybrid)
else:
    print("Run the Hugging Face section first to create a hybrid ranking.")

# Part D — Agentic AI

## 12. What makes the workflow agentic?

An agentic workflow is more than “ask an LLM for code.”

A useful loop is:

**GOAL → PLAN → TOOLS → OBSERVE → REVISE → VERIFY**

For this notebook:

- **Goal:** improve the job matcher;
- **Plan:** decompose one improvement into steps;
- **Tools:** Python, files, tests, Hugging Face, GitHub;
- **Observe:** inspect scores, rankings, errors, and tests;
- **Revise:** change code or assumptions;
- **Verify:** rerun tests and compare results.

The human remains responsible for acceptance criteria and final decisions.

## 13. Agentic task for Google Antigravity

Open the project/notebook in **Google Antigravity** and give the agent a bounded task.

### Copy/paste prompt

> We are building a teaching notebook for an AI-guided job-search application.  
> Before changing code, propose a short plan.  
> Improve only the `baseline_match()` and `explain_match()` portion of the notebook.  
> Requirements:  
> 1. Keep the score in `[0,1]`.  
> 2. Preserve separate evidence for skills, experience, interests, and constraints.  
> 3. Add at least three meaningful tests.  
> 4. Do not invent job facts not present in the input data.  
> 5. After editing, execute the tests and show the outputs.  
> 6. Summarize every change and list decisions that still require human approval.  
> Do not rewrite unrelated sections.

### Student observation log

Record the plan, cells/files changed, tests added, errors, recovery behavior, rejected suggestions, and remaining human edits.

## 14. Optional Hugging Face `smolagents` example

This demonstrates the *shape* of an agentic tool workflow with Hugging Face.

It is optional because model/API availability and free-tier limits can change.

### Rules

- Never hard-code a token in a notebook committed to GitHub.
- Store secrets in Colab Secrets or environment variables.
- Keep the tool narrow.
- Verify recommendations against actual job records.

In [ ]:
# OPTIONAL TEMPLATE — requires smolagents and an approved Hugging Face token/model.
#
# %pip -q install smolagents
#
# from google.colab import userdata
# os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
#
# from smolagents import CodeAgent, HfApiModel, tool
#
# @tool
# def lookup_job(job_id: str) -> str:
#     """Return one job record by job_id from the teaching dataset."""
#     matches = [j for j in jobs if j["job_id"] == job_id]
#     if not matches:
#         return "Job not found"
#     return json.dumps(matches[0], indent=2)
#
# model = HfApiModel(model_id="Qwen/Qwen2.5-Coder-32B-Instruct")
#
# agent = CodeAgent(
#     tools=[lookup_job],
#     model=model,
#     additional_authorized_imports=[]
# )
#
# result = agent.run(
#     "Inspect jobs J001, J006, and J008. "
#     "Using only facts returned by the tool, explain which best matches "
#     "a student with Python, SQL, pandas, machine learning, Git, healthcare interest, "
#     "and 2 years of experience. State uncertainty explicitly."
# )
#
# print(result)

print("Template only: configure an approved Hugging Face model/token before running.")

# Part E — GitHub Workflow

## 15. Make the experiment reproducible

Suggested repository structure:

```text
job-search-challenge/
├── README.md
├── notebooks/
│   └── challenge1_job_search.ipynb
├── data/
│   └── jobs_sample.csv
├── src/
│   └── matcher.py
├── tests/
│   └── test_matcher.py
└── results/
    └── comparison.csv
```

Suggested workflow:

1. **Issue** — task + acceptance criteria.
2. **Branch** — human baseline / AI-guided experiment.
3. **Commit** — small meaningful changes.
4. **PR / review** — explain AI contribution + human edits.
5. **README** — run instructions + results + limitations.

Suggested traceability labels: `HUMAN`, `AI-GENERATED`, `CO-DESIGNED`.

In [ ]:
# GitHub command examples.
# Replace placeholders before running.

# !git clone https://github.com/YOUR-ORG/YOUR-REPO.git
# %cd YOUR-REPO

# !git checkout -b human-baseline
# !git status
# !git add notebooks/challenge1_job_search.ipynb
# !git commit -m "HUMAN: add transparent baseline matcher"

# !git checkout -b ai-guided-matcher
# !git add .
# !git commit -m "AI-GENERATED: propose improved matching logic"
# !git commit -am "CO-DESIGNED: verify scores and fix explanation logic"

print("GitHub example commands are ready. Replace repository placeholders before use.")

## 16. Export the synthetic data for your repo

In [ ]:
export_df = jobs_df.copy()
export_df["required_skills"] = export_df["required_skills"].apply(json.dumps)
export_df["preferred_skills"] = export_df["preferred_skills"].apply(json.dumps)

csv_path = "jobs_sample.csv"
export_df.to_csv(csv_path, index=False)

print(f"Saved {csv_path} with {len(export_df)} synthetic job postings.")

# Part F — Manual vs. AI-Guided Evaluation

## 17. Compare versions with evidence

Do **not** assume AI should win.

Record the same evidence for manual and AI-guided versions:

- development time;
- tests passed;
- top-5 ranking quality;
- code clarity/modularity;
- transparency;
- reproducibility;
- human edits;
- main failure;
- lessons learned.

In [ ]:
comparison = pd.DataFrame([
    {
        "version": "Human baseline",
        "development_time_min": np.nan,
        "tests_passed": int(test_results["passed"].sum()),
        "tests_total": len(test_results),
        "top5_quality_1_to_5": np.nan,
        "human_edits": "Baseline written manually",
        "main_failure": "",
        "what_we_learned": ""
    },
    {
        "version": "AI-guided",
        "development_time_min": np.nan,
        "tests_passed": np.nan,
        "tests_total": np.nan,
        "top5_quality_1_to_5": np.nan,
        "human_edits": "",
        "main_failure": "",
        "what_we_learned": ""
    },
    {
        "version": "Human-AI co-designed",
        "development_time_min": np.nan,
        "tests_passed": np.nan,
        "tests_total": np.nan,
        "top5_quality_1_to_5": np.nan,
        "human_edits": "",
        "main_failure": "",
        "what_we_learned": ""
    }
])

comparison

## 18. Simple top-k evaluation

For a controlled classroom test, define a small expected-relevance set for the sample profile.

This is a **teaching device**, not universal ground truth.

In [ ]:
relevant_job_ids = {"J001", "J006", "J008", "J010"}

def precision_at_k(ranked_job_ids, relevant_ids, k=5):
    topk = ranked_job_ids[:k]
    return sum(j in relevant_ids for j in topk) / k

baseline_p5 = precision_at_k(
    baseline_ranked["job_id"].tolist(),
    relevant_job_ids,
    k=5
)

print("Baseline P@5:", round(baseline_p5, 3))

if hf_available:
    hybrid_p5 = precision_at_k(
        hybrid["job_id"].tolist(),
        relevant_job_ids,
        k=5
    )
    print("Hybrid P@5:", round(hybrid_p5, 3))

# Part G — Feedback & Refinement

## 19. User feedback loop

Users should be able to save/reject jobs, correct skills, adjust weights, refine queries, and explain why a result was wrong.

In [ ]:
feedback_log = []

def record_feedback(job_id, action, note="", corrected_skills=None):
    entry = {
        "timestamp": pd.Timestamp.utcnow().isoformat(),
        "job_id": job_id,
        "action": action,
        "note": note,
        "corrected_skills": corrected_skills or []
    }
    feedback_log.append(entry)
    return entry

record_feedback("J001", "save", "Strong healthcare + Python/SQL fit")
record_feedback("J003", "reject", "Experience requirement is too high")
record_feedback("J008", "save", "Good healthcare analytics fit")

pd.DataFrame(feedback_log)

## 20. Example refinement based on feedback

Suppose the user repeatedly rejects roles whose minimum experience exceeds current experience.

A co-designed improvement can add an explicit penalty rather than hide the preference inside a vague semantic score.

In [ ]:
def experience_gap_penalty(user_years, min_years, penalty_per_year=0.08):
    gap = max(min_years - user_years, 0)
    return min(gap * penalty_per_year, 0.30)

def refined_match(profile, job):
    result = baseline_match(profile, job)
    penalty = experience_gap_penalty(
        profile["years_experience"],
        job["min_experience"]
    )
    refined_score = max(result["score"] - penalty, 0.0)

    return {
        **result,
        "experience_gap_penalty": round(penalty, 4),
        "refined_score": round(refined_score, 4)
    }

refined_rows = []
for job in jobs:
    r = refined_match(user_profile, job)
    refined_rows.append({
        "job_id": job["job_id"],
        "title": job["title"],
        "company": job["company"],
        **r
    })

refined_ranked = (
    pd.DataFrame(refined_rows)
    .sort_values("refined_score", ascending=False)
    .reset_index(drop=True)
)

refined_ranked[[
    "job_id", "title", "company",
    "score", "experience_gap_penalty", "refined_score"
]]

# Part H — Responsible AI Checkpoints

## 21. Before you submit

### Privacy
Use synthetic or approved examples. Do not publish real resumes, contact information, or sensitive profile data.

### Bias
Inspect whether ranking overweights prestige, keywords, location, or other proxies that may disadvantage users.

### Hallucination
Separate facts present in the posting from model-generated interpretations or advice.

### Provenance
Record where job data came from and follow source/API terms.

### Human agency
Recommendations assist; they do not decide. Users should be able to change preferences, weights, and reject results.

### Uncertainty
Do not let an agent turn missing evidence into a confident claim.

# Part I — Challenge Tasks

## 22. Student tasks

### Task 1 — Human baseline
- Define one sample user profile.
- Add or replace the synthetic job data with the class-approved dataset.
- Run the transparent baseline.
- Create at least **5 test cases**.
- Record development time and pain points.

### Task 2 — AI-guided implementation
Use Antigravity or another class-approved agentic coding tool to improve **one bounded component**.

Possible components: skill normalization, title matching, explanation generation, test generation, ranking evaluation, feedback handling, or refactoring.

### Task 3 — Hugging Face enhancement
Add one Hugging Face model/component and explain what capability it adds, how you verify it, and when the simpler baseline is preferable.

### Task 4 — GitHub traceability
Show an issue, branch, meaningful commits, PR/review note, and README instructions.

### Task 5 — Compare
Include at least one example where AI helped and one where human review changed or rejected the AI proposal.

## 23. Reflection questions

1. Which decisions were easiest to automate?
2. Which decisions required the most human judgment?
3. Did the agent actually verify its changes or only say that it did?
4. Which ranking errors were caused by data representation?
5. Which were caused by scoring logic?
6. Did semantic matching improve relevance? At what cost to transparency?
7. What should be a hard constraint rather than a soft preference?
8. How would you evaluate fairness in a larger dataset?
9. What information should never be inferred if absent from a job posting?
10. If you had one more development cycle, what would you improve next?

# 24. Suggested deliverables

Your final Challenge 1 repository should contain:

- **Human design artifact** — requirements, architecture, baseline plan;
- **Working Colab/Jupyter notebook** — data → matching → ranking → explanation;
- **AI-guided development record** — prompts/tasks, agent changes, human review;
- **Comparison & evaluation** — manual vs. AI-guided results + failure analysis;
- **GitHub + reflection** — clean repo, README, reproducibility instructions, next steps.

**Success is not “AI wins.”**  
Success is a working prototype plus evidence that you understand how the design process changed when AI entered the loop.

# Part J — Streamlit Interactive Application

## 25. Move from notebook prototype to a web application

The notebook demonstrates the experimental workflow, while the companion **Streamlit application** turns the same job-search logic into an interactive user-facing prototype.

The Streamlit app includes:

- editable user profile;
- job search and filters;
- explainable ranking;
- matched and missing skills;
- adjustable score weights;
- optional Hugging Face semantic matching;
- hybrid baseline + semantic ranking;
- save/reject/needs-review feedback;
- manual vs. AI-guided comparison table;
- Google Antigravity agentic-development prompt;
- GitHub traceability workflow.

This preserves the Challenge 1 architecture:

**User Profile → Job Data → Match Engine → Results/Explanations → Feedback**

### Companion files

- `streamlit_app.py`
- `requirements.txt`
- `README_Streamlit.md`

### Run locally

```bash
pip install -r requirements.txt
streamlit run streamlit_app.py
```

### Recommended student experiment

1. Run the notebook baseline.
2. Run the Streamlit version.
3. Create a GitHub issue for one improvement.
4. Ask Antigravity to propose a plan before editing.
5. Let the agent change only the bounded component.
6. Run tests and inspect the Git diff.
7. Make human corrections.
8. Record the result as **HUMAN**, **AI-GENERATED**, or **CO-DESIGNED**.

In [ ]:
# Streamlit companion application
#
# Local launch:
#   pip install -r requirements.txt
#   streamlit run streamlit_app.py
#
# Inspect the companion app source from the notebook working directory:
from pathlib import Path

app_file = Path("streamlit_app.py")
if app_file.exists():
    print(app_file.read_text(encoding="utf-8")[:3000])
else:
    print("Place streamlit_app.py in the same project directory, then run: streamlit run streamlit_app.py")

## 26. Colab + Streamlit note

Google Colab is useful for the notebook experiments, while Streamlit is designed to run as a web application server. For the class, the cleanest workflow is:

- develop and test core functions in **Colab/Jupyter**;
- commit the notebook and `streamlit_app.py` to **GitHub**;
- run Streamlit locally or deploy it from the GitHub repository.

This keeps the notebook experiment and the interactive application reproducible without making the core exercise depend on a temporary notebook tunnel.